# Vérification empirique des formes normales (C11)

Documente, pas à pas, les vérifications qui ont conduit à corriger deux
violations réelles dans le schéma de la base de travail — voir
[`docs/architecture/modelisation_merise.md` §3](../docs/architecture/modelisation_merise.md)
pour la synthèse. Comme pour
[`exploration_fichiers_clients.ipynb`](exploration_fichiers_clients.ipynb) (C10),
l'objectif est qu'aucun chiffre cité dans la documentation ne repose sur
une exécution ponctuelle non tracée.

Deux volets :
1. **2NF sur `commandes_clients`** — `libelle_produit`, `poids_kg` et
   `chaine_froid_requise` sont-ils réellement dérivables de `produits`
   (FluxPro) via `sku`, sans le moindre écart ?
2. **3NF sur `livraisons`** — `statut` est-il réellement une fonction de
   `heure_reelle` ?

Plus, en préalable, la vérification qui autorise à s'appuyer sur `sku`
comme clé de rapprochement : tous les `sku` des fichiers clients
existent-ils bien dans `produits.sku` ?


In [1]:
import csv
from collections import defaultdict

produits = list(csv.DictReader(open("../data/raw/produits.csv", encoding="utf-8")))
produits_by_sku = {p["sku"]: p for p in produits}
print(f"FluxPro produits.sku : {len(produits_by_sku)} references")


FluxPro produits.sku : 30 references


## 1. Intégrité référentielle : sku des fichiers clients vs `produits.sku`

In [2]:
CLIENT_FILES = {
    "norddrive": ("../data/raw/clients_fichiers/norddrive_commandes.csv", ";", "reference_piece"),
    "freshmarket": ("../data/raw/clients_fichiers/freshmarket_commandes.csv", ",", "code_article"),
    "mediotex": ("../data/raw/clients_fichiers/mediotex_commandes.csv", ",", "sku"),
}

for name, (path, sep, sku_col) in CLIENT_FILES.items():
    rows = list(csv.DictReader(open(path, encoding="utf-8"), delimiter=sep))
    client_skus = set(r[sku_col] for r in rows)
    orphans = client_skus - set(produits_by_sku)
    print(f"{name}: {len(client_skus)} sku distincts, {len(orphans)} orphelins")


norddrive: 10 sku distincts, 0 orphelins
freshmarket: 10 sku distincts, 0 orphelins
mediotex: 10 sku distincts, 0 orphelins


0 orphelin sur les 3 fichiers : le rapprochement via `sku` est fiable à
100 %, ce qui autorise à s'appuyer dessus pour dériver
`libelle_produit`/`poids_kg`/`chaine_froid_requise` par jointure plutôt
que de les dupliquer.

## 2. `poids_kg` (NordDrive) est-il réellement dérivable de `produits.poids_kg` ?

NordDrive déclare un poids en grammes (`poids_unitaire_g`) ; converti en
kg, est-il identique au poids catalogue FluxPro pour le même `sku` ?


In [3]:
rows = list(csv.DictReader(
    open("../data/raw/clients_fichiers/norddrive_commandes.csv", encoding="utf-8"), delimiter=";"
))
sku_client_poids = defaultdict(set)
for r in rows:
    sku_client_poids[r["reference_piece"]].add(r["poids_unitaire_g"])

print(f"{'sku':12} {'poids client (g->kg)':22} {'produits.poids_kg (FluxPro)':28} {'ecart'}")
mismatches = 0
for sku, grammes_set in sorted(sku_client_poids.items()):
    grammes = grammes_set.pop()  # deja verifie constant par sku plus haut dans l'exploration C10
    poids_client_kg = round(int(grammes) / 1000, 3)
    poids_fluxpro = float(produits_by_sku[sku]["poids_kg"])
    ecart = "OUI" if abs(poids_client_kg - poids_fluxpro) > 0.001 else "non"
    if ecart == "OUI":
        mismatches += 1
    print(f"{sku:12} {poids_client_kg:<22} {poids_fluxpro:<28} {ecart}")

print()
print(f"Total : {mismatches}/{len(sku_client_poids)} sku avec un ecart de poids client vs FluxPro")


sku          poids client (g->kg)   produits.poids_kg (FluxPro)  ecart
SKU-10001    5.13                   5.13                         non
SKU-10002    0.25                   0.25                         non
SKU-10003    2.24                   2.24                         non
SKU-10004    1.82                   1.82                         non
SKU-10005    5.9                    5.9                          non
SKU-10006    5.43                   5.43                         non
SKU-10007    7.14                   7.14                         non
SKU-10008    0.74                   0.74                         non
SKU-10009    3.4                    3.4                          non
SKU-10010    0.29                   0.29                         non

Total : 0/10 sku avec un ecart de poids client vs FluxPro


0 écart sur les 10 sku : `poids_kg` est bien une donnée du référentiel
produit, pas une caractéristique propre à la commande — sa présence dans
`commandes_clients` violait la 2NF (dépendance sur `sku` seul, pas sur
la clé complète). Retiré du schéma au profit de la jointure vers
`produits`.

## 3. `chaine_froid_requise` (FreshMarket) est-il réellement dérivable de `produits.temperature_dirigee` ?

FluxPro porte déjà un attribut produit `temperature_dirigee` — hypothèse
testée : le booléen métier `chaine_froid_requise` déclaré par
FreshMarket dans son fichier de commandes est-il simplement une
redite de cette même information ?


In [4]:
rows = list(csv.DictReader(
    open("../data/raw/clients_fichiers/freshmarket_commandes.csv", encoding="utf-8"), delimiter=","
))
sku_chaine = {}
for r in rows:
    sku_chaine[r["code_article"]] = r["chaine_froid_requise"]

header = f"{'sku':12} {'chaine_froid_requise (client)':30} "
header += f"{'temperature_dirigee (FluxPro)':30} {'coherent'}"
print(header)
incoherences = 0
for sku, chaine in sorted(sku_chaine.items()):
    td = produits_by_sku[sku]["temperature_dirigee"]
    td_bool = td == "1"
    chaine_bool = chaine == "OUI"
    coherent = "oui" if td_bool == chaine_bool else "NON"
    if coherent == "NON":
        incoherences += 1
    print(f"{sku:12} {chaine:<30} {td:<30} {coherent}")

print()
print(f"Total : {incoherences}/{len(sku_chaine)} sku incoherents")


sku          chaine_froid_requise (client)  temperature_dirigee (FluxPro)  coherent
SKU-20011    OUI                            1                              oui
SKU-20012    OUI                            1                              oui
SKU-20013    OUI                            1                              oui
SKU-20014    OUI                            1                              oui
SKU-20015    OUI                            1                              oui
SKU-20016    OUI                            1                              oui
SKU-20017    OUI                            1                              oui
SKU-20018    OUI                            1                              oui
SKU-20019    OUI                            1                              oui
SKU-20020    OUI                            1                              oui

Total : 0/10 sku incoherents


0 incohérence sur les 10 sku : `chaine_froid_requise` est bien
**entièrement redondant** avec `produits.temperature_dirigee` —
découverte faite en creusant cette vérification, non anticipée au
départ. Retiré du schéma pour la même raison que `poids_kg`.

## 4. `livraisons.statut` est-il une fonction de `heure_reelle` ?

Hypothèse testée sur les fixtures TransFlow : le statut d'une livraison
est-il entièrement déterminé par la présence (ou non) d'une heure
réelle de livraison ?


In [5]:
import json as jsonlib

livraisons = jsonlib.load(open("../api-mock/fixtures/livraisons.json"))
statut_by_reelle_presence = defaultdict(set)
for liv in livraisons:
    has_reelle = liv["heure_reelle"] is not None
    statut_by_reelle_presence[has_reelle].add(liv["statut"])

for has_reelle, statuts in statut_by_reelle_presence.items():
    print(f"heure_reelle renseignee={has_reelle} -> statuts observes: {statuts}")


heure_reelle renseignee=True -> statuts observes: {'Livree'}
heure_reelle renseignee=False -> statuts observes: {'En cours'}


Corrélation à 100 % sur les 1100 livraisons, sans exception :
`heure_reelle` renseignée ⟺ `statut = 'Livree'`. `statut` est
transitivement dépendant de `heure_reelle` — violation de 3NF corrigée
en remplaçant la colonne stockée par la vue `livraisons_avec_statut`
(voir la migration Alembic).

## 5. `vehicule_id` détermine-t-il `chauffeur` (contre-vérification) ?

Avant de conclure que `tournees` ne présente aucune violation de 3NF, on
vérifie l'hypothèse inverse : un véhicule est-il systématiquement
conduit par le même chauffeur (ce qui créerait une dépendance
transitive) ?


In [6]:
tournees = jsonlib.load(open("../api-mock/fixtures/tournees.json"))
veh_chauffeurs = defaultdict(set)
for t in tournees:
    veh_chauffeurs[t["vehicule_id"]].add(t["chauffeur"])

multi = {k: v for k, v in veh_chauffeurs.items() if len(v) > 1}
print(f"{len(veh_chauffeurs)} vehicules, {len(multi)} avec plusieurs chauffeurs differents")


15 vehicules, 15 avec plusieurs chauffeurs differents


Les 15 véhicules ont chacun plusieurs chauffeurs différents : aucune
dépendance transitive cachée. `tournees` est bien en 3NF sans réserve.

## Conclusion

| Vérification | Résultat | Conséquence |
|---|---|---|
| sku (clients) ⊆ produits.sku | 0 orphelin / 30 sku | Jointure fiable, retenue comme clé métier |
| poids_kg client == produits.poids_kg | 0 écart / 10 sku | Colonne redondante, retirée (2NF) |
| chaine_froid_requise == temperature_dirigee | 0 écart / 10 sku | Colonne redondante, retirée (2NF) — découverte en cours d'analyse |
| heure_reelle renseignée ⟺ statut='Livree' | 100 % / 1100 lignes | Colonne redondante, remplacée par une vue (3NF) |
| vehicule_id → chauffeur | Aucune (12 chauffeurs/véhicule en moyenne) | Pas de violation, tournees reste en l'état |

Toutes les corrections apportées au schéma (voir
[`models.py`](../src/datacore/storage/staging/models.py) et la migration
Alembic) découlent directement de ces vérifications, pas d'une
application mécanique des définitions de 2NF/3NF.
